# IoT Botnet Intrusion Classification with a From-Scratch Random Forest (N-BaIoT)

**Tanishq Nayak (23BCP098)** · Computer Science and Engineering, PDEU · 7th Semester ML Project-Based Learning

This notebook contains the complete machine-learning code of the project. Every learning component is implemented
from scratch with numpy (**no scikit-learn / ML libraries**): data splitting and scaling, a CART decision tree, a
Random Forest, PCA, and all evaluation metrics (confusion matrix, precision/recall/F1, ROC/AUC).

| # | Section |
|---|---|
| 1 | Setup and configuration |
| 2 | Dataset: loading and labelling (N-BaIoT, Danmini doorbell) |
| 3 | Stratified train / validation / test split and train-only scaling |
| 4 | From-scratch algorithms: decision tree, random forest, PCA, metrics |
| 5 | Dataset visualisation |
| 6 | Final model: training and evaluation |
| 7 | Overfitting / underfitting analysis |
| 8 | Stricter evaluation: time-ordered split |
| 9 | Streaming 115-feature extractor (for the live Raspberry Pi demo) |
| 10 | Exporting the model for the Pi, and how to run the live pipeline |

**How to run.** Put the 11 Danmini CSV files (`benign_traffic.csv`, `ack.csv`, `scan_1.csv`, `syn.csv`, `udp_1.csv`,
`combo.csv`, `junk.csv`, `scan.csv`, `tcp.csv`, `udp.csv`, `udpplain.csv`) in `data/raw/` of the repository (see `data/README.md`) or set the
`NBAIOT_DIR` environment variable. Outputs (figures, exported model) are written to `results/notebook/`. A full run takes roughly 10 minutes (the sweeps and the time-ordered evaluation dominate).
Setting the environment variable `PBL_FAST=1` runs everything on a small subset in a couple of minutes, for a quick smoke test
(numbers will then differ from the reported ones).

## 1. Setup and configuration

In [ ]:
import os, time, math
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

FAST = os.environ.get("PBL_FAST", "0") == "1"                  # quick smoke-test mode
RAW_DIR = Path(os.environ.get("NBAIOT_DIR", "../data/raw"))    # folder with the 11 N-BaIoT Danmini CSVs (relative to notebooks/)
OUT_DIR = Path("../results/notebook"); OUT_DIR.mkdir(parents=True, exist_ok=True)   # figures / exported model go here
RANDOM_STATE = 42
SAMPLE_PER_CLASS = 1000 if FAST else 5000                      # rows per class used in the working subset
N_TREES, MAX_DEPTH = (10, 10) if FAST else (30, 15)            # final model (chosen from the sweeps in section 7)
plt.rcParams["figure.dpi"] = 100
print(f"FAST mode: {FAST} | data folder: {RAW_DIR.resolve()} | rows/class: {SAMPLE_PER_CLASS} | forest: {N_TREES} trees, depth {MAX_DEPTH}")

## 2. Dataset: loading and labelling

N-BaIoT contains traffic from nine commercial IoT devices infected with the **Mirai** and **BASHLITE (Gafgyt)** botnets. We use the
**Danmini doorbell**: one benign file, four Mirai attack files and six Gafgyt attack files, giving **11 classes**. Each row has
**115 statistical features** (5 traffic streams x 5 decay windows x their statistics). 5,000 rows per class are sampled so the
classes are balanced (the raw data is heavily imbalanced; benign is under 5% of rows).

In [ ]:
# file name -> (botnet family, attack subtype)
FILE_LABELS = {
    "benign_traffic.csv": ("benign", "benign"),
    "ack.csv": ("mirai", "ack"), "scan_1.csv": ("mirai", "scan"),
    "syn.csv": ("mirai", "syn"), "udp_1.csv": ("mirai", "udp"),
    "combo.csv": ("gafgyt", "combo"), "junk.csv": ("gafgyt", "junk"),
    "scan.csv": ("gafgyt", "scan"), "tcp.csv": ("gafgyt", "tcp"),
    "udp.csv": ("gafgyt", "udp"), "udpplain.csv": ("gafgyt", "udpplain"),
}
N_FEATURES = 115


def load_labeled_dataset(sample_per_class, random_state=42):
    """Load every file, tag rows with a combined class label (e.g. 'mirai_ack', 'benign'), concatenate."""
    frames = []
    for fname, (botnet, subtype) in FILE_LABELS.items():
        df = pd.read_csv(RAW_DIR / fname, nrows=30000 if FAST else None)
        if sample_per_class is not None and len(df) > sample_per_class:
            df = df.sample(n=sample_per_class, random_state=random_state)
        df["label"] = subtype if botnet == "benign" else f"{botnet}_{subtype}"
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


df = load_labeled_dataset(SAMPLE_PER_CLASS, RANDOM_STATE)
feature_cols = df.columns[:N_FEATURES].tolist()
print("shape:", df.shape, "| missing values:", int(df[feature_cols].isnull().sum().sum()))
print(df["label"].value_counts().to_string())
df[feature_cols[:6] + ["label"]].head()

## 3. Stratified train / validation / test split and train-only scaling

80% train+validation / 20% test, then 80/20 of the former, i.e. **64% / 16% / 20%**, stratified by class and implemented manually.
Z-score scaling statistics are computed on the **training set only** and reused for validation and test (no leakage).

In [ ]:
def stratified_split(frame, label_col, test_frac, random_state=42):
    """Per class: shuffle indices, send test_frac of them to the right split, the rest to the left split."""
    rng = np.random.default_rng(random_state)
    left, right = [], []
    for _, group in frame.groupby(label_col):
        idx = group.index.to_numpy().copy()
        rng.shuffle(idx)
        n_right = int(round(len(idx) * test_frac))
        right.append(frame.loc[idx[:n_right]])
        left.append(frame.loc[idx[n_right:]])
    shuffle = lambda parts: pd.concat(parts).sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    return shuffle(left), shuffle(right)


def zscore_fit(X):
    mean, std = X.mean(axis=0), X.std(axis=0)
    std[std == 0] = 1.0                      # guard constant columns
    return mean, std


def zscore_apply(X, mean, std):
    return (X - mean) / std


train_val_df, test_df = stratified_split(df, "label", 0.20, RANDOM_STATE)
train_df, val_df = stratified_split(train_val_df, "label", 0.20, RANDOM_STATE)
for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"{name:11s}: {len(part):>6,} rows ({len(part)/len(df):.0%})")

X_train_raw = train_df[feature_cols].to_numpy(dtype=np.float64)
X_val_raw = val_df[feature_cols].to_numpy(dtype=np.float64)
X_test_raw = test_df[feature_cols].to_numpy(dtype=np.float64)
y_train, y_val, y_test = (d["label"].to_numpy() for d in (train_df, val_df, test_df))

scaler_mean, scaler_std = zscore_fit(X_train_raw)            # fitted on TRAIN only
X_train = zscore_apply(X_train_raw, scaler_mean, scaler_std)
X_val = zscore_apply(X_val_raw, scaler_mean, scaler_std)
X_test = zscore_apply(X_test_raw, scaler_mean, scaler_std)
print(f"train mean after scaling: {X_train.mean():+.2e}  std: {X_train.std():.3f}  (validation mean {X_val.mean():+.3f}: scaled with TRAIN statistics)")

## 4. From-scratch algorithms

### 4.1 Decision tree (CART, Gini impurity)
Binary splits on numeric features minimising weighted Gini impurity. The split search is **vectorised**: samples are sorted once per feature and
cumulative class counts give the impurity of every candidate threshold without Python loops over samples.

In [ ]:
"""
From-scratch CART Decision Tree Classifier (Gini impurity, binary numeric splits).
Only numpy is used. Split search is vectorised: samples are sorted once per
feature and class counts for every candidate threshold come from a cumulative sum.
"""
import numpy as np


class TreeNode:
    __slots__ = ["feature_idx", "threshold", "left", "right", "prediction", "is_leaf"]

    def __init__(self):
        self.feature_idx = self.threshold = self.left = self.right = self.prediction = None
        self.is_leaf = False


def gini_impurity(y):
    """Gini = 1 - sum(p_c^2)."""
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return 1.0 - np.sum(p ** 2)


class DecisionTreeClassifier:
    """max_features: None | int | float | "sqrt" -> features tried per split
    (the source of extra randomness when used inside a Random Forest)."""

    def __init__(self, max_depth=10, min_samples_split=10, min_samples_leaf=5,
                 max_features=None, random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state
        self.root = None
        self.classes_ = None
        self._rng = np.random.default_rng(random_state)

    def _n_features_to_use(self, n):
        if self.max_features is None:
            return n
        if self.max_features == "sqrt":
            return max(1, int(np.sqrt(n)))
        if isinstance(self.max_features, float):
            return max(1, int(self.max_features * n))
        return min(self.max_features, n)

    def _best_split(self, X, y, feature_indices):
        """Return (feature, threshold, gini) minimising weighted Gini, or None.
        Candidate thresholds: midpoints if <=32 unique values, else 32 quantiles."""
        best_gini, best_feat, best_thr = np.inf, None, None
        n = X.shape[0]
        _, y_int = np.unique(y, return_inverse=True)
        n_classes = y_int.max() + 1

        for f in feature_indices:
            values = X[:, f]
            uniq = np.unique(values)
            if len(uniq) <= 1:
                continue
            if len(uniq) > 32:
                cands = np.unique(np.quantile(values, np.linspace(0, 1, 33)[1:-1]))
            else:
                cands = (uniq[:-1] + uniq[1:]) / 2.0
            if len(cands) == 0:
                continue

            order = np.argsort(values)
            sorted_vals, sorted_y = values[order], y_int[order]
            one_hot = np.zeros((n, n_classes), dtype=np.int64)
            one_hot[np.arange(n), sorted_y] = 1
            cum = np.cumsum(one_hot, axis=0)          # left-side class counts
            total = cum[-1]
            positions = np.searchsorted(sorted_vals, cands, side="right")

            for pos, thr in zip(positions, cands):
                n_l, n_r = pos, n - pos
                if n_l < self.min_samples_leaf or n_r < self.min_samples_leaf:
                    continue
                lc = cum[pos - 1]
                rc = total - lc
                g = (n_l / n) * (1 - np.sum((lc / n_l) ** 2)) + (n_r / n) * (1 - np.sum((rc / n_r) ** 2))
                if g < best_gini:
                    best_gini, best_feat, best_thr = g, f, thr

        return None if best_feat is None else (best_feat, best_thr, best_gini)

    @staticmethod
    def _majority(y):
        vals, counts = np.unique(y, return_counts=True)
        return vals[np.argmax(counts)]

    def _build(self, X, y, depth):
        node = TreeNode()
        n, n_feat = X.shape
        if (len(np.unique(y)) == 1 or n < self.min_samples_split
                or (self.max_depth is not None and depth >= self.max_depth)):
            node.is_leaf, node.prediction = True, self._majority(y)
            return node

        feats = self._rng.choice(n_feat, size=self._n_features_to_use(n_feat), replace=False)
        split = self._best_split(X, y, feats)
        if split is None:
            node.is_leaf, node.prediction = True, self._majority(y)
            return node

        node.feature_idx, node.threshold, _ = split
        mask = X[:, node.feature_idx] <= node.threshold
        node.left = self._build(X[mask], y[mask], depth + 1)
        node.right = self._build(X[~mask], y[~mask], depth + 1)
        return node

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.root = self._build(X, y, 0)
        return self

    def _predict_one(self, x, node):
        while not node.is_leaf:
            node = node.left if x[node.feature_idx] <= node.threshold else node.right
        return node.prediction

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])

### 4.2 Random Forest
Bootstrap sampling of rows (bagging) + random subset of sqrt(115) = 10 features at every split + majority vote. The fraction of trees voting for a class is its score for the ROC analysis.

In [ ]:
"""
From-scratch Random Forest: bootstrap aggregating (bagging) over rows +
per-split feature subsampling (inside DecisionTreeClassifier), majority vote.
Uses only numpy and our own DecisionTreeClassifier.
"""
import numpy as np
from collections import Counter


class RandomForestClassifier:
    def __init__(self, n_estimators=50, max_depth=10, min_samples_split=10,
                 min_samples_leaf=5, max_features="sqrt", bootstrap_frac=1.0,
                 random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.bootstrap_frac = bootstrap_frac
        self.random_state = random_state
        self.trees = []
        self.classes_ = None

    def fit(self, X, y, verbose=True):
        rng = np.random.default_rng(self.random_state)
        n = X.shape[0]
        n_boot = int(round(n * self.bootstrap_frac))
        self.classes_ = np.unique(y)
        self.trees = []
        for i in range(self.n_estimators):
            idx = rng.integers(0, n, size=n_boot)              # sample WITH replacement
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth, min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf, max_features=self.max_features,
                random_state=int(rng.integers(0, 2**31 - 1)))
            self.trees.append(tree.fit(X[idx], y[idx]))
            if verbose and (i + 1) % max(1, self.n_estimators // 10) == 0:
                print(f"  Trained tree {i + 1}/{self.n_estimators}")
        return self

    def predict(self, X):
        votes = np.array([t.predict(X) for t in self.trees])    # (n_trees, n_samples)
        return np.array([Counter(votes[:, c]).most_common(1)[0][0] for c in range(votes.shape[1])])

    def predict_proba(self, X):
        """Fraction of trees voting for each class, columns ordered by classes_."""
        votes = np.array([t.predict(X) for t in self.trees])
        idx = {c: i for i, c in enumerate(self.classes_)}
        proba = np.zeros((X.shape[0], len(self.classes_)))
        for c in range(X.shape[0]):
            for cls, k in Counter(votes[:, c]).items():
                proba[c, idx[cls]] = k / self.n_estimators
        return proba

### 4.3 PCA (eigendecomposition of the covariance matrix)

In [ ]:
"""
Principal Component Analysis from scratch (eigendecomposition of the covariance
matrix with numpy; no sklearn.decomposition). Fit on TRAIN only, then applied to val/test.
"""
import numpy as np


class PCA:
    def __init__(self, n_components):
        self.n_components = n_components
        self.mean_ = self.components_ = None
        self.explained_variance_ = self.explained_variance_ratio_ = None
        self.all_eigvals_ = None

    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        cov = np.cov(X - self.mean_, rowvar=False)
        eigvals, eigvecs = np.linalg.eigh(cov)          # symmetric matrix -> eigh
        order = np.argsort(eigvals)[::-1]                # largest variance first
        eigvals, eigvecs = eigvals[order], eigvecs[:, order]
        self.components_ = eigvecs[:, :self.n_components].T
        self.explained_variance_ = eigvals[:self.n_components]
        self.explained_variance_ratio_ = self.explained_variance_ / eigvals.sum()
        self.all_eigvals_ = eigvals
        return self

    def transform(self, X):
        return (X - self.mean_) @ self.components_.T

    def fit_transform(self, X):
        return self.fit(X).transform(X)

### 4.4 Evaluation metrics (confusion matrix, precision/recall/F1, one-vs-rest ROC and AUC)

In [ ]:
_trapz = getattr(np, "trapezoid", None) or np.trapz     # numpy 2.x / 1.x compatibility


def confusion_matrix(y_true, y_pred, classes):
    idx = {c: i for i, c in enumerate(classes)}
    cm = np.zeros((len(classes), len(classes)), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[idx[t], idx[p]] += 1
    return cm                                   # rows = true, cols = predicted


def precision_recall_f1(cm, classes):
    rows = []
    for i, c in enumerate(classes):
        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * p * r / (p + r) if p + r else 0.0
        rows.append({"class": c, "precision": p, "recall": r, "f1": f1, "support": int(cm[i].sum())})
    return pd.DataFrame(rows)


def roc_curve_ovr(y_bin, scores, n_thresholds=100):
    thr = np.linspace(1.0, 0.0, n_thresholds)
    P = y_bin.sum(); N = len(y_bin) - P
    tpr = np.array([np.sum((scores >= t) & (y_bin == 1)) / P if P else 0.0 for t in thr])
    fpr = np.array([np.sum((scores >= t) & (y_bin == 0)) / N if N else 0.0 for t in thr])
    return fpr, tpr


def auc(fpr, tpr):
    o = np.argsort(fpr)
    return float(_trapz(tpr[o], fpr[o]))

In [ ]:
# quick self-checks of the building blocks on tiny synthetic data
rng = np.random.default_rng(0)
Xd = rng.normal(size=(300, 5)); yd = (Xd[:, 0] + Xd[:, 1] - Xd[:, 2] > 0).astype(int)
print("tree accuracy on synthetic data:", (DecisionTreeClassifier(max_depth=6, random_state=0).fit(Xd, yd).predict(Xd) == yd).mean())
print("forest accuracy on synthetic data:", (RandomForestClassifier(n_estimators=10, max_depth=6, random_state=0).fit(Xd, yd, verbose=False).predict(Xd) == yd).mean())

## 5. Dataset visualisation

In [ ]:
CLASSES = sorted(np.unique(y_train))
COLOR = dict(zip(CLASSES, plt.cm.tab20(np.linspace(0, 1, len(CLASSES)))))

# 5.1 class distribution
counts = df["label"].value_counts().reindex(CLASSES)
fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(counts.index, counts.values, color=[COLOR[c] for c in counts.index])
ax.set(title="Class distribution of the working sample (balanced by design)", ylabel="samples")
plt.xticks(rotation=45, ha="right")
for b in bars:
    ax.annotate(int(b.get_height()), (b.get_x() + b.get_width() / 2, b.get_height()), ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR / "class_distribution.png", dpi=150); plt.show()

In [ ]:
# 5.2 correlation heatmap of a 30-feature sample spread over all five stream blocks
idx = np.linspace(0, N_FEATURES - 1, 30).astype(int)
cols = [feature_cols[i] for i in idx]
corr = df[cols].corr().to_numpy()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(30)); ax.set_yticks(range(30))
ax.set_xticklabels(cols, rotation=90, fontsize=6); ax.set_yticklabels(cols, fontsize=6)
ax.set_title("Feature correlation (30 of 115 features): the same statistic over different windows is highly redundant")
fig.colorbar(im, ax=ax, label="Pearson correlation")
plt.tight_layout(); plt.savefig(OUT_DIR / "correlation_heatmap.png", dpi=150); plt.show()

In [ ]:
# 5.3 PCA projection (fitted on the training set) and 5.4 cumulative explained variance
pca2 = PCA(2); X2 = pca2.fit_transform(X_train)
fig, ax = plt.subplots(figsize=(9, 7))
for c in CLASSES:
    m = y_train == c
    ax.scatter(X2[m, 0], X2[m, 1], s=6, alpha=0.5, color=COLOR[c], label=c)
r = pca2.explained_variance_ratio_
ax.set(title=f"PCA projection of training data ({r.sum()*100:.1f}% of variance in 2 components)",
       xlabel=f"PC1 ({r[0]*100:.1f}%)", ylabel=f"PC2 ({r[1]*100:.1f}%)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, markerscale=3)
plt.tight_layout(); plt.savefig(OUT_DIR / "pca_scatter.png", dpi=150); plt.show()

pca_full = PCA(50).fit(X_train)
cum = np.cumsum(pca_full.all_eigvals_[:50]) / pca_full.all_eigvals_.sum()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(range(1, 51), cum, marker="o", markersize=3); ax.axhline(0.95, color="red", ls="--", lw=1, label="95% variance")
ax.set(xlabel="principal components", ylabel="cumulative explained variance", title="PCA scree plot"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "pca_scree.png", dpi=150); plt.show()
for k in (2, 5, 10, 20, 30, 50):
    print(f"{k:>3} components -> {cum[k-1]*100:6.2f}% of the variance")

## 6. Final model: training and evaluation on the held-out test set
Random Forest with 30 trees, maximum depth 15, minimum split size 10, minimum leaf size 5 (chosen from the validation sweeps in section 7).

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=N_TREES, max_depth=MAX_DEPTH, min_samples_split=10,
                            min_samples_leaf=5, max_features="sqrt", random_state=RANDOM_STATE)
rf.fit(X_train, y_train, verbose=False)
print(f"training time: {time.time() - t0:.1f}s")

classes = rf.classes_
y_pred, proba = rf.predict(X_test), rf.predict_proba(X_test)
print(f"train accuracy:      {(rf.predict(X_train) == y_train).mean():.4f}")
print(f"validation accuracy: {(rf.predict(X_val) == y_val).mean():.4f}")
print(f"TEST accuracy:       {(y_pred == y_test).mean():.4f}")

cm = confusion_matrix(y_test, y_pred, classes)
prf = precision_recall_f1(cm, classes)
print(f"macro precision / recall / F1: {prf.precision.mean():.4f} / {prf.recall.mean():.4f} / {prf.f1.mean():.4f}")
errors = [(str(classes[i]), str(classes[j]), int(cm[i, j])) for i in range(len(classes)) for j in range(len(classes)) if i != j and cm[i, j] > 0]
print("misclassifications (true -> predicted, count):", errors)

In [ ]:
# confusion matrix heat-map
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes, rotation=60, ha="right", fontsize=8); ax.set_yticklabels(classes, fontsize=8)
for i in range(len(classes)):
    for j in range(len(classes)):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7, color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set(xlabel="predicted", ylabel="true", title="Confusion matrix (test set)")
plt.tight_layout(); plt.savefig(OUT_DIR / "confusion_matrix.png", dpi=150); plt.show()

# one-vs-rest ROC curves and AUC
fig, ax = plt.subplots(figsize=(7, 6))
auc_rows = []
for i, c in enumerate(classes):
    fpr, tpr = roc_curve_ovr((y_test == c).astype(int), proba[:, i])
    auc_rows.append({"class": c, "auc": auc(fpr, tpr)})
    ax.plot(fpr, tpr, label=f"{c} ({auc_rows[-1]['auc']:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set(xlabel="false positive rate", ylabel="true positive rate", title="One-vs-rest ROC curves (test set)")
ax.legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.savefig(OUT_DIR / "roc_curves.png", dpi=150); plt.show()

result_table = prf.merge(pd.DataFrame(auc_rows), on="class")
result_table.round(4)

## 7. Overfitting / underfitting analysis
Train versus **validation** accuracy as model complexity grows: tree depth (single tree and forest) and number of trees. A large gap would signal
overfitting; low accuracy on both sets signals underfitting. (The test set is not used for these decisions.)

In [ ]:
def run_sweep(title, key, values, make_model, fit_kwargs=None):
    rows = []
    for v in values:
        t0 = time.time()
        m = make_model(v).fit(X_train, y_train, **(fit_kwargs or {}))
        tr, va = (m.predict(X_train) == y_train).mean(), (m.predict(X_val) == y_val).mean()
        rows.append({key: "None" if v is None else v, "train_acc": tr, "val_acc": va, "gap": tr - va, "seconds": round(time.time() - t0, 1)})
    out = pd.DataFrame(rows)
    print(f"--- {title} ---"); print(out.round(4).to_string(index=False))
    return out


depths = [1, 2, 5, 8, None] if FAST else [1, 2, 3, 5, 8, 10, 15, 20, None]
sweep_tree = run_sweep("single tree vs max_depth", "depth", depths,
                       lambda d: DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE))
sweep_forest = run_sweep("forest vs max_depth", "depth", depths,
                         lambda d: RandomForestClassifier(n_estimators=5 if FAST else 20, max_depth=d, random_state=RANDOM_STATE),
                         {"verbose": False})
sweep_trees = run_sweep("forest (depth 10) vs number of trees", "n_estimators", [1, 3, 5] if FAST else [1, 5, 10, 20, 30, 50],
                        lambda n: RandomForestClassifier(n_estimators=n, max_depth=10, random_state=RANDOM_STATE),
                        {"verbose": False})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (sw, key, ttl) in zip(axes, [(sweep_tree, "depth", "Single tree"), (sweep_forest, "depth", "Forest"),
                                     (sweep_trees, "n_estimators", "Forest (depth 10)")]):
    xs = range(len(sw))
    ax.plot(xs, sw["train_acc"], marker="o", label="train"); ax.plot(xs, sw["val_acc"], marker="s", label="validation")
    ax.set_xticks(list(xs)); ax.set_xticklabels(sw[key].astype(str))
    ax.set(title=f"{ttl}: accuracy vs {key}", xlabel=key, ylabel="accuracy"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(OUT_DIR / "overfitting_sweeps.png", dpi=150); plt.show()

**Reading the sweeps.** (1) At depth 1-2 a tree has only 2-4 leaves, so it can predict at most 2-4 of the 11 classes: accuracy is about 2/11 and 4/11 on *both* train and
validation, the signature of **underfitting** (high bias). (2) The forest recovers much faster at shallow depth because its trees see different samples and feature subsets.
(3) From depth ~10 both models exceed 99.5% with a train-validation gap of at most ~0.2 percentage points, and depth 15 / 20 / unlimited are identical because every leaf is already
pure: there is no sign of classical **overfitting** on this random split. (4) More than ~30 trees brings little gain. Section 8 shows why a random split can still be too optimistic.

## 8. Stricter evaluation: time-ordered split
N-BaIoT features are exponentially decayed running statistics, so consecutive rows are near-duplicates. A random split can put a row's neighbours in the training set and the row itself
in the test set. Here, for every class, training rows come from the **first 65%** of its capture and test rows from the **last 30%** (5% gap); scaling is fitted on the training data only.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n_tr, n_te = (600, 300) if FAST else (4000, 1500)
Xtr_t, ytr_t, Xte_t, yte_t = [], [], [], []
for fname, (botnet, subtype) in FILE_LABELS.items():
    arr = pd.read_csv(RAW_DIR / fname, nrows=30000 if FAST else None).to_numpy(dtype=np.float64)
    n = len(arr)
    train_pool, test_pool = arr[: int(n * 0.65)], arr[int(n * 0.70):]
    label = subtype if botnet == "benign" else f"{botnet}_{subtype}"
    Xtr_t.append(train_pool[rng.choice(len(train_pool), n_tr, replace=False)]); ytr_t += [label] * n_tr
    Xte_t.append(test_pool[rng.choice(len(test_pool), n_te, replace=False)]); yte_t += [label] * n_te
Xtr_t, Xte_t, ytr_t, yte_t = np.vstack(Xtr_t), np.vstack(Xte_t), np.array(ytr_t), np.array(yte_t)
m_t, s_t = zscore_fit(Xtr_t)
rf_t = RandomForestClassifier(n_estimators=N_TREES, max_depth=MAX_DEPTH, min_samples_split=10, min_samples_leaf=5,
                              max_features="sqrt", random_state=RANDOM_STATE).fit(zscore_apply(Xtr_t, m_t, s_t), ytr_t, verbose=False)
pred_t = rf_t.predict(zscore_apply(Xte_t, m_t, s_t))
cm_t = confusion_matrix(yte_t, pred_t, rf_t.classes_)
prf_t = precision_recall_f1(cm_t, rf_t.classes_)

acc_random, acc_time = (y_pred == y_test).mean(), (pred_t == yte_t).mean()
print(f"random split accuracy: {acc_random:.4f}   |   time-ordered split accuracy: {acc_time:.4f}")
family = lambda c: "benign" if c == "benign" else c.split("_")[0]
fam_acc = np.mean([family(t) == family(p) for t, p in zip(yte_t, pred_t)])
print(f"family-level (benign / mirai / gafgyt) accuracy on the time-ordered split: {fam_acc:.4f}")
print(prf_t.round(3).to_string(index=False))
errs = [(str(rf_t.classes_[i]), str(rf_t.classes_[j]), int(cm_t[i, j])) for i in range(len(cm_t)) for j in range(len(cm_t)) if i != j and cm_t[i, j] > 50]
print("main confusions (more than 50 rows):", errs)

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.bar(["random split", "time-ordered split"], [acc_random, acc_time], color=["#4c72b0", "#dd8452"])
ax.set(ylim=(0, 1.05), ylabel="test accuracy", title="Optimism of a random split")
for x, v in enumerate([acc_random, acc_time]):
    ax.text(x, v + 0.01, f"{v:.3f}", ha="center")
plt.tight_layout(); plt.savefig(OUT_DIR / "random_vs_time_split.png", dpi=150); plt.show()

**Reference results from the full run (reported in the project report):** random split 99.90%, time-ordered split 82.27%. The loss is concentrated in four Gafgyt classes
(`gafgyt_junk` predicted as `gafgyt_combo`, `gafgyt_udp` predicted as `gafgyt_tcp`), while benign traffic and the Mirai / Gafgyt families remain separable (99.98% family-level accuracy).
A plausible explanation (a hypothesis, not yet proven) is that these floods look alike once an attack run is under way. Block-wise cross-validation, merging the confusable classes, or sampling
training windows from the whole capture are the next things to try.

## 9. Streaming 115-feature extractor (for the live demo)
N-BaIoT publishes only pre-computed features, not packets, so a live system must recompute them itself. The extractor below re-implements the **damped incremental statistics** (AfterImage, Mirsky et al., NDSS 2018):
each stream keeps exponentially decayed sums (weight `w`, `CF1`, `CF2`) with decay `2^(-lambda * dt)`, giving mean = CF1/w and variance = |CF2/w - mean^2|; linked directional streams also keep a decayed residual-product sum for covariance / correlation.
Five streams x five windows give the 115 features in the dataset's column order. It was validated against the authors' reference implementation (maximum relative deviation 9e-13 over 4,000 packets x 115 features; script `validate_extractor.py`).

In [ ]:
"""
Damped incremental statistics (the "AfterImage" engine from Kitsune, Mirsky et al.,
NDSS 2018) - our own implementation, validated numerically against the authors'
reference code (see validate_extractor.py).

Each stream keeps exponentially-decayed sums:  w (weight), CF1 (sum), CF2 (sum of squares)
Decay before each insertion:  gamma = 2^(-lambda * dt)   (lambda in 1/seconds)
  mean = CF1/w          var = |CF2/w - mean^2|          std = sqrt(var)
Two streams can be linked by an IncStatCov, which keeps a decayed sum of residual
products (CF3, w3) -> approximate covariance cov = CF3/w3 and pcc = cov/(std_i*std_j).
A packet updates ONLY its own direction's stream; the partner stream is merely decayed
to the same time and its last residual is reused (this is what makes it streaming-safe).

Numerical fidelity: like the reference, mean/var/std are CACHED and invalidated only when a
new value is inserted. Decay scales CF1, CF2 and w together so mean/var are mathematically
unchanged, but recomputing var = |CF2/w - mean^2| after decay adds floating-point noise that
blows up pcc for near-constant streams (the dataset shows pcc == 0 for 75-99% of rows, i.e. the
exact-zero behaviour of the cached version).

Engineering additions for long-running use (do not change results in normal operation):
  * weights are floored at 1e-300 to avoid ZeroDivisionError after a very long idle gap
  * StatDB.prune() removes dead streams so memory stays bounded on a Raspberry Pi
"""
import math

_W_FLOOR = 1e-300


class IncStat:
    """One damped 1-D stream (e.g. packet sizes sent by one host)."""
    __slots__ = ("id", "lam", "cf1", "cf2", "w", "is_diff", "last_t", "covs",
                 "_mean", "_var", "_std")

    def __init__(self, lam, stream_id, init_time=0.0, is_diff=False):
        self.id = stream_id
        self.lam = lam
        self.cf1 = 0.0
        self.cf2 = 0.0
        self.w = 1e-20            # tiny non-zero weight avoids divide-by-zero on first read
        self.is_diff = is_diff    # jitter stream: the inserted value is the inter-arrival time
        self.last_t = init_time
        self.covs = []            # IncStatCov trackers this stream participates in
        self._mean = self._var = self._std = None   # caches, invalidated by insert()

    def decay(self, t):
        dt = t - self.last_t
        if dt > 0:                # out-of-order / identical timestamps do not decay
            f = 2.0 ** (-self.lam * dt)
            self.cf1 *= f
            self.cf2 *= f
            self.w = max(self.w * f, _W_FLOOR)
            self.last_t = t       # (cached mean/var/std stay valid: decay scales all sums equally)

    def insert(self, v, t):
        if self.is_diff:
            d = t - self.last_t   # time since this stream's previous packet
            v = d if d > 0 else 0.0
        self.decay(t)
        self.cf1 += v
        self.cf2 += v * v
        self.w += 1.0
        self._mean = self._var = self._std = None   # force recomputation
        for c in self.covs:       # keep linked covariance trackers in sync
            c.update(self.id, v, t)

    def mean(self):
        if self._mean is None:
            self._mean = self.cf1 / self.w
        return self._mean

    def var(self):
        if self._var is None:
            self._var = abs(self.cf2 / self.w - self.mean() ** 2)
        return self._var

    def std(self):
        if self._std is None:
            self._std = math.sqrt(self.var())
        return self._std

    def stats_1d(self):
        """[weight, mean, variance] - refreshes mean/var caches like the reference's allstats_1D()."""
        self._mean = self.cf1 / self.w
        self._var = abs(self.cf2 / self.w - self._mean ** 2)
        return self.w, self._mean, self._var


class IncStatCov:
    """Links two streams; tracks decayed sum of residual products (approx. covariance)."""
    __slots__ = ("s", "last_res", "cf3", "w3", "last_t")

    def __init__(self, s0, s1, init_time=0.0):
        self.s = (s0, s1)
        self.last_res = [0.0, 0.0]
        self.cf3 = 0.0
        self.w3 = 1e-20
        self.last_t = init_time

    def update(self, stream_id, v, t):
        """Call AFTER stream `stream_id` has already absorbed (v, t)."""
        if stream_id == self.s[0].id:
            k = 0
        elif stream_id == self.s[1].id:
            k = 1
        else:
            raise ValueError("stream id not part of this covariance tracker")
        other = 1 - k
        self.s[other].decay(t)            # bring partner stream to the current time
        self._decay(t, k)
        res = v - self.s[k].mean()        # residual of this packet vs its stream mean
        self.cf3 += res * self.last_res[other]
        self.w3 += 1.0
        self.last_res[k] = res

    def _decay(self, t, k):
        dt = t - self.last_t
        if dt > 0:
            f = 2.0 ** (-self.s[k].lam * dt)
            self.cf3 *= f
            self.w3 = max(self.w3 * f, _W_FLOOR)
            self.last_t = t
            self.last_res[k] *= f

    def cov(self):
        return self.cf3 / self.w3

    def pcc(self):
        ss = self.s[0].std() * self.s[1].std()
        return self.cov() / ss if ss != 0 else 0.0


class StatDB:
    """Hash table of streams keyed by (stream_id, lambda)."""

    def __init__(self):
        self.ht = {}

    def _get(self, sid, lam, t, is_diff=False):
        key = (sid, lam)
        s = self.ht.get(key)
        if s is None:
            s = self.ht[key] = IncStat(lam, sid, t, is_diff)
        return s

    def update_1d(self, sid, t, v, lam, is_diff=False):
        """Insert v, return [weight, mean, variance]."""
        s = self._get(sid, lam, t, is_diff)
        s.insert(v, t)
        return s.stats_1d()

    def update_1d2d(self, id1, id2, t, v, lam):
        """Insert v into stream id1 (partner id2 = opposite direction); return
        [weight, mean, std, magnitude, radius, covariance, pcc]."""
        s1 = self._get(id1, lam, t)
        s1.insert(v, t)
        s2 = self._get(id2, lam, t)
        cov = None
        for c in s1.covs:
            if c.s[0].id == id2 or c.s[1].id == id2:
                cov = c
                break
        if cov is None:
            cov = IncStatCov(s1, s2, t)
            s1.covs.append(cov)
            s2.covs.append(cov)
        cov.update(id1, v, t)   # (reference quirk kept: tracker is updated here as well)
        w1, m1, v1 = s1.stats_1d()                      # refresh s1's caches (as reference does)
        magnitude = math.sqrt(s1.mean() ** 2 + s2.mean() ** 2)
        radius = math.sqrt(s1.var() ** 2 + s2.var() ** 2)
        return w1, m1, math.sqrt(v1), magnitude, radius, cov.cov(), cov.pcc()

    def prune(self, now, min_weight=1e-3):
        """Drop streams whose decayed weight (at time `now`) is negligible."""
        dead = [k for k, s in self.ht.items()
                if s.w * 2.0 ** (-s.lam * max(0.0, now - s.last_t)) < min_weight]
        for k in dead:
            s = self.ht.pop(k)
            for c in s.covs:                       # unlink from partner streams
                for o in c.s:
                    if o is not s and c in o.covs:
                        o.covs.remove(c)
        return len(dead)

In [ ]:
"""
Streaming 115-feature extractor in the N-BaIoT column layout:
  blocks  MI_dir | H | HH | HH_jit | HpHp ,  inside each block: lambda L5,L3,L1,L0.1,L0.01,
  inside each lambda: the block's statistics.

  MI_dir  src MAC+IP stream            -> weight, mean, variance
  H       src IP stream                -> weight, mean, variance
  HH      src->dst IP channel (+ reverse direction as partner)
                                       -> weight, mean, std, magnitude, radius, covariance, pcc
  HH_jit  inter-arrival times on the src->dst channel -> weight, mean, variance
  HpHp    src ip:port -> dst ip:port socket (+ reverse) -> same 7 stats as HH

Feature keying of HH/HpHp (per directional channel/socket) is inferred from the dataset
(HH_weight <= H_weight, equal on ~21% of rows); see validate_extractor.py.
"""
import numpy as np

LAMBDAS = (5, 3, 1, 0.1, 0.01)
LNAMES = ("L5", "L3", "L1", "L0.1", "L0.01")
STATS_1D = ("weight", "mean", "variance")
STATS_2D = ("weight", "mean", "std", "magnitude", "radius", "covariance", "pcc")
BLOCKS = (("MI_dir", STATS_1D), ("H", STATS_1D), ("HH", STATS_2D),
          ("HH_jit", STATS_1D), ("HpHp", STATS_2D))

FEATURE_NAMES = [f"{b}_{ln}_{s}" for b, stats in BLOCKS for ln in LNAMES for s in stats]
assert len(FEATURE_NAMES) == 115


class FeatureExtractor:
    def __init__(self, prune_every=5000):
        self.db = {name: StatDB() for name, _ in BLOCKS}
        self._n = 0
        self._prune_every = prune_every

    def update(self, ts, size, src_mac, dst_mac, src_ip, dst_ip, src_proto, dst_proto):
        """Process one packet, return the 115-dim feature vector (np.ndarray).
        size = frame length in bytes; src_proto/dst_proto = port (str) or 'arp'."""
        out = np.empty(115)
        pos = 0
        mi, h, hh, jit, hphp = (self.db[n] for n, _ in BLOCKS)
        mi_id, h_id = src_mac + src_ip, src_ip
        hh_a, hh_b = f"{src_ip}>{dst_ip}", f"{dst_ip}>{src_ip}"
        jit_id = src_ip + dst_ip
        if src_proto == "arp":
            sk_a, sk_b = src_mac, dst_mac
        else:
            sk_a, sk_b = f"{src_ip}:{src_proto}>{dst_ip}:{dst_proto}", f"{dst_ip}:{dst_proto}>{src_ip}:{src_proto}"

        blocks_out = {name: [] for name, _ in BLOCKS}   # collect per block, then lay out in CSV order
        for lam in LAMBDAS:
            blocks_out["MI_dir"].extend(mi.update_1d(mi_id, ts, size, lam))
            blocks_out["H"].extend(h.update_1d(h_id, ts, size, lam))
            blocks_out["HH"].extend(hh.update_1d2d(hh_a, hh_b, ts, size, lam))
            blocks_out["HH_jit"].extend(jit.update_1d(jit_id, ts, 0.0, lam, is_diff=True))
            blocks_out["HpHp"].extend(hphp.update_1d2d(sk_a, sk_b, ts, size, lam))
        for name, _ in BLOCKS:
            vals = blocks_out[name]
            out[pos:pos + len(vals)] = vals
            pos += len(vals)

        self._n += 1
        if self._n % self._prune_every == 0:
            for d in self.db.values():
                d.prune(ts)
        return out

In [ ]:
# 1) feature names and order must be identical to the dataset's CSV columns
print("feature names / order identical to the dataset columns:", FEATURE_NAMES == feature_cols)

# 2) run the extractor on a small synthetic packet stream and benchmark it
rng = np.random.default_rng(7)
hosts = [("aa:00:00:00:00:0%d" % i, "10.0.0.%d" % (i + 1)) for i in range(5)]
def synthetic_packets(n):
    t, out = 1000.0, []
    for _ in range(n):
        t += rng.exponential(0.02)
        a, b = rng.choice(len(hosts), 2, replace=False)
        out.append((t, float(rng.integers(60, 1500)), hosts[a][0], hosts[b][0], hosts[a][1], hosts[b][1],
                    str(rng.choice([80, 443, 5000])), str(rng.choice([80, 443, 5000]))))
    return out

pk = synthetic_packets(5000 if FAST else 20000)
fx = FeatureExtractor()
t0 = time.time(); vecs = np.array([fx.update(*p) for p in pk]); dt = time.time() - t0
print(f"{len(pk):,} packets -> feature matrix {vecs.shape}, all finite: {bool(np.isfinite(vecs).all())}, {len(pk)/dt:,.0f} packets/s")

## 10. Exporting the model for the Raspberry Pi
The trained forest is flattened into plain arrays and saved with the scaler in one `.npz` file; the Pi then needs **only numpy** (no training code, no pickle).
`FlatForest` predicts exactly like the original forest.

In [ ]:
"""
Export a trained from-scratch Random Forest (+ scaler) to a single .npz file and load it
back as a lightweight predictor that needs ONLY numpy (ideal for the Raspberry Pi: no pickle,
no training code). Trees are flattened into parallel arrays; leaves have cls >= 0.
"""
import numpy as np


def _flatten(root, class_index, offset):
    feat, thr, left, right, cls = [], [], [], [], []

    def add(node):
        i = len(feat)
        feat.append(-1); thr.append(0.0); left.append(-1); right.append(-1); cls.append(-1)
        if node.is_leaf:
            cls[i] = class_index[node.prediction]
        else:
            feat[i] = int(node.feature_idx)
            thr[i] = float(node.threshold)
            left[i] = add(node.left) + offset
            right[i] = add(node.right) + offset
        return i

    add(root)
    return feat, thr, left, right, cls


def export_forest(rf, path, scaler_mean, scaler_std):
    class_index = {c: i for i, c in enumerate(rf.classes_)}
    F, T, L, R, C, roots = [], [], [], [], [], []
    for tree in rf.trees:
        off = len(F)
        roots.append(off)
        f, t, l, r, c = _flatten(tree.root, class_index, off)
        F += f; T += t; L += l; R += r; C += c
    np.savez_compressed(path, feat=np.array(F, np.int32), thr=np.array(T), left=np.array(L, np.int32),
                        right=np.array(R, np.int32), cls=np.array(C, np.int32),
                        roots=np.array(roots, np.int32), classes=np.array(rf.classes_, dtype=str),
                        mean=np.asarray(scaler_mean), std=np.asarray(scaler_std))


class FlatForest:
    """Inference-only forest. predict_one() takes a RAW (unscaled) 115-feature vector."""

    def __init__(self, path):
        z = np.load(path, allow_pickle=False)
        # plain Python lists: scalar indexing is much faster than on numpy arrays
        self.feat, self.thr = z["feat"].tolist(), z["thr"].tolist()
        self.left, self.right, self.cls = z["left"].tolist(), z["right"].tolist(), z["cls"].tolist()
        self.roots = z["roots"].tolist()
        self.classes = [str(c) for c in z["classes"]]
        self.mean, self.std = z["mean"], z["std"]

    def predict_proba_one(self, x_raw):
        x = ((x_raw - self.mean) / self.std).tolist()
        votes = [0] * len(self.classes)
        feat, thr, left, right, cls = self.feat, self.thr, self.left, self.right, self.cls
        for i in self.roots:
            while cls[i] < 0:
                i = left[i] if x[feat[i]] <= thr[i] else right[i]
            votes[cls[i]] += 1
        n = len(self.roots)
        return [v / n for v in votes]

    def predict_one(self, x_raw):
        p = self.predict_proba_one(x_raw)
        return self.classes[max(range(len(p)), key=p.__getitem__)]

In [ ]:
export_forest(rf, OUT_DIR / "n_baiot_model.npz", scaler_mean, scaler_std)
flat = FlatForest(OUT_DIR / "n_baiot_model.npz")
same = np.mean([flat.predict_one(X_test_raw[i]) == y_pred[i] for i in range(min(500, len(y_pred)))])
t0 = time.time(); [flat.predict_one(x) for x in X_test_raw[:500]]
print(f"exported model reproduces the forest's predictions on {same:.1%} of 500 test rows | {500/(time.time()-t0):,.0f} predictions/s")

### Running the live pipeline (command line, outside the notebook)
Files (in `src/`): `packet_features.py`, `live_detector.py`, `train_live_model.py`, `make_test_pcaps.py`, `model_io.py`, `afterimage.py`, `extractor.py`. Run the commands below from the repository root.

```
# try everything without hardware, using synthetic traffic
python3 src/make_test_pcaps.py --out pcaps/train --seed 1
python3 src/live_detector.py collect --pcap pcaps/train/mixed.pcap --out data/live/session1.csv \
        --schedule "benign:6,syn_flood:6,benign:6,udp_flood:6,benign:6,port_scan:6,benign:6"
python3 src/train_live_model.py --data data/live --out models/live_model.npz --final
python3 src/live_detector.py detect --pcap pcaps/train/mixed.pcap --model models/live_model.npz

# on the Raspberry Pi (own isolated network only; needs sudo and scapy)
sudo python3 src/live_detector.py collect --iface wlan0 --schedule "benign:120,syn_flood:30,benign:60" --out data/live/s1.csv
sudo python3 src/live_detector.py detect  --iface wlan0 --model models/live_model.npz
```
**Lesson for data collection:** the long-window features remember history, so record *continuous* sessions labelled by time segment (`--schedule`) rather than one cold-start file per class.
Synthetic traffic is far easier to separate than real traffic; it only demonstrates that the pipeline works.